<a href="https://colab.research.google.com/github/ilyeoncz/Finance-Assistant-/blob/main/Financeassistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
!pip install google-genai
from google.colab import userdata
from google import genai
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-3.8-flash"  # Using 2.5-flash as the standard stable model
import gradio as gr

print("Connected to the Gemini API.")
print("Welcome to your personalised finance assistant!")

from datetime import datetime
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print(f"Session started: {timestamp}")
print()

# ==========================================
# 1. Terminal Input for Theme Customization
# ==========================================
user_color_choice = input("What color theme would you like for your finance tracker? (Options: green, blue, rose, violet): ").strip().lower()

color_map = {
    "green": "green",
    "blue": "blue",
    "rose": "rose",
    "violet": "violet"
}

# Default to "rose" if they type something unexpected
selected_hue = color_map.get(user_color_choice, "rose")
print(f"Applying theme color: {selected_hue.capitalize()}...\n")


# ==========================================
# 2. Core Helper Functions (Modules 3 & 5)
# ==========================================

def calculate_percentage(amount, total):
    # figure out what percentage this specific expense is of the total budget
    if total == 0:
        return 0
    return (amount / total) * 100

def categorize_expense(amount, monthly_budget):
    # tag the expense impact level based on how much of the budget it eats up
    percentage = calculate_percentage(amount, monthly_budget)

    if percentage >= 20:
        return "High"
    elif percentage >= 5:
        return "Moderate"
    else:
        return "Minimal"

def calculate_total_spent(transactions):
    # loop through our saved items and add up the total cost
    if not transactions:
        return 0.0
    return sum(t['amount'] for t in transactions)


# ==========================================
# 3. Dictionary Storage & Pandas (Modules 6 & 7)
# ==========================================

# list to hold all our transaction dictionaries
user_transactions = []

def build_pandas_dataframe(transactions):
    # turn our list of dictionaries into a clean pandas table so we can do math easily
    if not transactions:
        return pd.DataFrame(columns=["Description", "Amount", "Category", "Expense Level"])

    data = []
    for t in transactions:
        data.append({
            "Description": t["description"],
            "Amount": t["amount"],
            "Category": t["category"],
            "Expense Level": t["level"]
        })
    return pd.DataFrame(data)


# ==========================================
# 4. Gemini AI Integration (Modules 1 & 8)
# ==========================================

def generate_ai_feedback(df, budget, total_spent):
    # pass our actual pandas table text into gemini using the pre-initialized client
    try:
        prompt = (
            f"Here is my actual expense log:\n{df.to_string(index=False)}\n\n"
            f"My monthly budget is ${budget:.2f} and I've spent ${total_spent:.2f} so far. "
            f"Look at my spending, especially anything marked High or Moderate, "
            f"and give me 2-3 short, practical tips to manage my money better."
        )

        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={"system_instruction": "You are a helpful, realistic personal finance coach."}
        )
        return response.text
    except Exception as e:
        return f"API Error: {str(e)}"


# ==========================================
# 5. Main Tracker Logic
# ==========================================

def process_finance_tracker(description, amount, category, monthly_budget):
    # basic error check so empty inputs or negative numbers don't crash the app
    if not description or description.strip() == "":
        return "Error: Please write a description.", "", pd.DataFrame()

    try:
        num_amount = float(amount)
        budget = float(monthly_budget)
        if num_amount < 0 or budget <= 0:
            raise ValueError("Amounts must be positive.")
    except ValueError as e:
        return f"Input Error: {str(e)}", "", pd.DataFrame()

    # figure out expense impact level
    level = categorize_expense(num_amount, budget)

    # store the new entry as a dictionary
    new_transaction = {
        "description": description.strip(),
        "amount": num_amount,
        "category": category.lower(),
        "level": level
    }
    user_transactions.append(new_transaction)

    # convert to pandas dataframe
    df = build_pandas_dataframe(user_transactions)

    total_spent = df["Amount"].sum()
    remaining_budget = budget - total_spent
    percentage_used = calculate_percentage(total_spent, budget)

    summary_text = (
        f"--- Budget Overview ---\n"
        f"• Total Budget: ${budget:.2f}\n"
        f"• Total Spent: ${total_spent:.2f} ({percentage_used:.1f}% used)\n"
        f"• Remaining: ${remaining_budget:.2f}\n"
        f"• Items Logged: {len(df)}"
    )

    # get custom advice from gemini using our data table
    ai_feedback = generate_ai_feedback(df, budget, total_spent)

    return summary_text, ai_feedback, df


# ==========================================
# 6. Testing Suite (Requirement R5)
# ==========================================

def run_unit_tests():
    global user_transactions
    user_transactions = [] # clear out old data

    # test our helper functions
    assert calculate_percentage(50, 500) == 10.0, "Percentage math failed"
    assert categorize_expense(120, 500) == "High", "Categorization failed for high amount"
    assert categorize_expense(10, 500) == "Minimal", "Categorization failed for small amount"

    # test adding to dictionary and dataframe conversion
    user_transactions.append({"description": "Test Item", "amount": 10.0, "category": "food", "level": "Minimal"})
    df_test = build_pandas_dataframe(user_transactions)
    assert len(df_test) == 1, "DataFrame length is wrong"
    assert calculate_total_spent(user_transactions) == 10.0, "Total sum calculation failed"

    print("All unit tests passed successfully!")

run_unit_tests()
print()


# ==========================================
# 7. Gradio Web Interface (Requirement R4)
# ==========================================

with gr.Blocks() as demo:
    gr.Markdown("# 💰 Personal Finance Assistant")
    gr.Markdown(f"Custom Theme Loaded: **{selected_hue.capitalize()}**")
    gr.Markdown("Track your spending, check your budget with pandas, and get automated AI coaching.")

    with gr.Row():
        with gr.Column():
            desc_input = gr.Textbox(label="Expense Description", placeholder="e.g., Groceries")
            amount_input = gr.Number(label="Amount ($)", value=0.0)
            category_input = gr.Dropdown(
                choices=["Food & Dining", "Education", "Bills & Utilities", "Leisure", "Transport", "General"],
                label="Category",
                value="General"
            )
            budget_input = gr.Number(label="Monthly Budget ($)", value=500.0)

            submit_btn = gr.Button("Log Expense & Analyze", variant="primary")

        with gr.Column():
            output_summary = gr.Textbox(label="Summary", lines=5)
            output_ai = gr.Textbox(label="AI Coach Feedback", lines=8)

    output_table = gr.Dataframe(label="Transaction History")

    submit_btn.click(
        fn=process_finance_tracker,
        inputs=[desc_input, amount_input, category_input, budget_input],
        outputs=[output_summary, output_ai, output_table]
    )


# ==========================================
# 8. Launch Web App
# ==========================================
if __name__ == "__main__":
    demo.launch(theme=gr.themes.Soft(primary_hue=selected_hue))

Connected to the Gemini API.
Welcome to your personalised finance assistant!
Session started: 2026-10-20 06:21:22

What color theme would you like for your finance tracker? (Options: green, blue, rose, violet): violet
Applying theme color: Violet...

All unit tests passed successfully!

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c008d8cc34ee605781.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
